<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/Aula%2025%20-%20%C3%81rvores%20de%20Comportamento%20code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Aula 25 — Árvores de Comportamento (*Behavior Trees*) a 50 Hz no SCADA-Core
### Linha Automatizada de Envase e Inspeção de Bebidas — Grupo 3 (ECAA08)

**Objetivo:** Implementar o motor de decisão reativo baseado em pulso de **Tick ($50\text{ Hz}$ / $\Delta t = 20\text{ ms}$)** com retorno ternário (`SUCCESS`, `FAILURE`, `RUNNING`) e nós compostos **`Selector (?)`** e **`Sequence (→)`**, garantindo preempção determinística na linha de envase:

$$\text{SIS Crítico (P1)} \succ \text{Obstrução VS2 (P2)} \succ \text{Pressurização RRT AS1 (P3)} \succ \text{Descarte Qualidade (P4)} \succ \text{Envase Nominal (P5)}$$

In [6]:
from typing import List, Callable

SUCCESS = "SUCCESS"
FAILURE = "FAILURE"
RUNNING = "RUNNING"

class BTNode:
    def tick(self, bb: dict) -> str:
        raise NotImplementedError

class ConditionNode(BTNode):
    def __init__(self, name: str, fn: Callable[[dict], bool]):
        self.name = name
        self.fn = fn

    def tick(self, bb: dict) -> str:
        res = SUCCESS if self.fn(bb) else FAILURE
        bb["trace"].append((self.name, res))
        return res

class ActionNode(BTNode):
    def __init__(self, name: str, fn: Callable[[dict], str]):
        self.name = name
        self.fn = fn

    def tick(self, bb: dict) -> str:
        res = self.fn(bb)
        bb["trace"].append((self.name, res))
        return res

class SequenceNode(BTNode):
    def __init__(self, name: str, children: List[BTNode]):
        self.name = name
        self.children = children

    def tick(self, bb: dict) -> str:
        for ch in self.children:
            status = ch.tick(bb)
            if status != SUCCESS:
                return status
        return SUCCESS

class SelectorNode(BTNode):
    def __init__(self, name: str, children: List[BTNode]):
        self.name = name
        self.children = children

    def tick(self, bb: dict) -> str:
        for ch in self.children:
            status = ch.tick(bb)
            if status != FAILURE:
                return status
        return FAILURE

print("✅ Motor de Behavior Tree (Selector, Sequence, Condition, Action) inicializado.")

✅ Motor de Behavior Tree (Selector, Sequence, Condition, Action) inicializado.


### 1. Montagem da Behavior Tree da Linha de Envase
Cada ramo lê diretamente as variáveis instrumentadas (norma **ANSI/ISA-5.1**) armazenadas no *Blackboard* (`e_stop`, `sp1`, `sp2`, `sq2`, `vs2_aberta`, `sl1_pct`, `sfc1_tampa`, `linha_habilitada`) e atualiza o comando dos atuadores (`BC1`, `VS1`, `VS2`, `VALV_Alivio` e esteira `RC1`).

In [7]:
def acao_trip_sis(bb: dict) -> str:
    alivio = 1 if bb["sp2"] >= 4.5 else 0
    bb.update({
        "cmd": "TRIP_SIS_PARADA",
        "rc1_vel": 0.0,
        "atuadores": f"BC1=0, VS1=0, VS2=0, VALV_Alivio={alivio}, RC1=STOP"
    })
    return SUCCESS

def acao_obstrucao_vs2(bb: dict) -> str:
    bb.update({
        "cmd": "PAUSA_OBSTRUCAO_VS2",
        "rc1_vel": 0.0,
        "atuadores": "BC1=1, VS1=1, VS2=0, RC1=PAUSA (POP-SEC-02)"
    })
    return RUNNING

def acao_pressurizacao_rrt(bb: dict) -> str:
    bb.update({
        "cmd": "PARTIDA_RRT_AS1",
        "rc1_vel": 0.1,
        "atuadores": "BC1=1, VS1=1, VS2=0, RC1=LENTA (Rampa AS1)"
    })
    return RUNNING

def acao_descarte_qualidade(bb: dict) -> str:
    motivo = "NIVEL<95%" if bb["sl1_pct"] < 95.0 else "TAMPA_AUSENTE"
    bb.update({
        "cmd": f"DESCARTE_REJ ({motivo})",
        "rc1_vel": 0.3,
        "atuadores": "BC1=1, VS1=1, VS2=1, RC1=DESVIO_REJEITO"
    })
    return RUNNING

def acao_envase_nominal(bb: dict) -> str:
    bb.update({
        "cmd": "PROD_APROV_NOMINAL",
        "rc1_vel": 0.4,
        "atuadores": "BC1=1, VS1=1, VS2=1, RC1=PALETIZACAO"
    })
    return RUNNING

bt_envase = SelectorNode("Root_Selector_SCADA", [
    # Ramo P1: Segurança Funcional Crítica (R-09 e_stop, R-04 SP2 >= 4.5 Barg, R-03 SP1 <= 1.0 Barg)
    SequenceNode("Seq_P1_SIS_Critico", [
        ConditionNode("Cond_SIS_Critico", lambda bb: bb["e_stop"] or bb["sp2"] >= 4.5 or bb["sp1"] <= 1.0),
        ActionNode("Act_Trip_SIS", acao_trip_sis)
    ]),
    # Ramo P2: Obstrução no Bico de Envase VS2 (R-05: SQ2 <= 2.0 L/min com VS2 aberta)
    SequenceNode("Seq_P2_Obstrucao_VS2", [
        ConditionNode("Cond_Obstrucao_VS2", lambda bb: bb["vs2_aberta"] and bb["sq2"] <= 2.0),
        ActionNode("Act_Pausa_Limpeza_VS2", acao_obstrucao_vs2)
    ]),
    # Ramo P3: Pressurização Controlada do Acumulador AS1 (RRT — SP2 < 3.0 Barg)
    SequenceNode("Seq_P3_Pressurizacao_RRT", [
        ConditionNode("Cond_Subpressao_AS1", lambda bb: bb["sp2"] < 3.0),
        ActionNode("Act_Partida_RRT_AS1", acao_pressurizacao_rrt)
    ]),
    # Ramo P4: Controle de Qualidade (SL1 < 95% ou SFC1 = 0 -> DESCARTE_REJ)
    SequenceNode("Seq_P4_Qualidade", [
        ConditionNode("Cond_Falha_Qualidade", lambda bb: bb["sl1_pct"] < 95.0 or not bb["sfc1_tampa"]),
        ActionNode("Act_Desvio_Rejeito", acao_descarte_qualidade)
    ]),
    # Ramo P5: Operação Nominal de Envase e Paletização
    SequenceNode("Seq_P5_Envase_Nominal", [
        ConditionNode("Cond_Linha_Habilitada", lambda bb: bb["linha_habilitada"]),
        ActionNode("Act_Envase_Nominal", acao_envase_nominal)
    ])
])

print("✅ Behavior Tree da Linha de Envase (SCADA-Core) montada com 5 ramos prioritários.")

✅ Behavior Tree da Linha de Envase (SCADA-Core) montada com 5 ramos prioritários.


### 2. Simulação de 6 Ciclos de *Tick* ($50\text{ Hz}$) sob Diferentes Estados do *Blackboard*
Testamos a preempção hierárquica em cenários reais da planta: operação nominal, rejeição de garrafa, partida com `AS1` despressurizado, obstrução do bico `VS2`, trip crítico por sobrepressão em `AS1` (mesmo com falha de nível simultânea) e linha em espera segura (`IDLE`).

In [8]:
cenarios_bt = [
    {"nome": "Tick 1 (Nominal)",            "e_stop": False, "sp1": 2.5, "sp2": 3.8, "sq2": 5.0, "vs2_aberta": True,  "sl1_pct": 98.0, "sfc1_tampa": True,  "linha_habilitada": True},
    {"nome": "Tick 2 (Nível Baixo SL1)",    "e_stop": False, "sp1": 2.5, "sp2": 3.8, "sq2": 5.0, "vs2_aberta": True,  "sl1_pct": 91.0, "sfc1_tampa": True,  "linha_habilitada": True},
    {"nome": "Tick 3 (Partida AS1 < 3.0)",  "e_stop": False, "sp1": 2.4, "sp2": 2.2, "sq2": 0.0, "vs2_aberta": False, "sl1_pct": 98.0, "sfc1_tampa": True,  "linha_habilitada": True},
    {"nome": "Tick 4 (Obstrução VS2)",      "e_stop": False, "sp1": 2.5, "sp2": 3.7, "sq2": 1.4, "vs2_aberta": True,  "sl1_pct": 98.0, "sfc1_tampa": True,  "linha_habilitada": True},
    {"nome": "Tick 5 (Sobrepressão AS1)",   "e_stop": False, "sp1": 2.5, "sp2": 4.7, "sq2": 1.4, "vs2_aberta": True,  "sl1_pct": 90.0, "sfc1_tampa": False, "linha_habilitada": True},
    {"nome": "Tick 6 (Linha Desabilitada)", "e_stop": False, "sp1": 2.5, "sp2": 3.5, "sq2": 0.0, "vs2_aberta": False, "sl1_pct": 98.0, "sfc1_tampa": True,  "linha_habilitada": False},
]

resultados_bt = []
for c in cenarios_bt:
    bb = dict(c, trace=[], cmd="IDLE_ESPERA", rc1_vel=0.0, atuadores="Todos em espera")
    status = bt_envase.tick(bb)
    resultados_bt.append((c["nome"], status, bb["cmd"], bb["rc1_vel"], len(bb["trace"])))
    print(f"{c['nome']:<28} -> Status={status:<8} | Comando={bb['cmd']:<26} | RC1={bb['rc1_vel']:.1f} m/s | Nós={len(bb['trace'])}")

Tick 1 (Nominal)             -> Status=RUNNING  | Comando=PROD_APROV_NOMINAL         | RC1=0.4 m/s | Nós=6
Tick 2 (Nível Baixo SL1)     -> Status=RUNNING  | Comando=DESCARTE_REJ (NIVEL<95%)   | RC1=0.3 m/s | Nós=5
Tick 3 (Partida AS1 < 3.0)   -> Status=RUNNING  | Comando=PARTIDA_RRT_AS1            | RC1=0.1 m/s | Nós=4
Tick 4 (Obstrução VS2)       -> Status=RUNNING  | Comando=PAUSA_OBSTRUCAO_VS2        | RC1=0.0 m/s | Nós=3
Tick 5 (Sobrepressão AS1)    -> Status=SUCCESS  | Comando=TRIP_SIS_PARADA            | RC1=0.0 m/s | Nós=2
Tick 6 (Linha Desabilitada)  -> Status=FAILURE  | Comando=IDLE_ESPERA                | RC1=0.0 m/s | Nós=5


### 3. Suíte de Asserções Automatizadas (Validação de Preempção e Curto-Circuito)

In [9]:
assert resultados_bt[0][1] == RUNNING and resultados_bt[0][2] == "PROD_APROV_NOMINAL", "Tick 1 deve operar em modo nominal"
assert resultados_bt[1][1] == RUNNING and "DESCARTE_REJ" in resultados_bt[1][2], "Tick 2 deve rejeitar garrafa com nível < 95%"
assert resultados_bt[2][1] == RUNNING and resultados_bt[2][2] == "PARTIDA_RRT_AS1", "Tick 3 deve acionar rampa RRT do acumulador AS1"
assert resultados_bt[3][1] == RUNNING and resultados_bt[3][2] == "PAUSA_OBSTRUCAO_VS2", "Tick 4 deve pausar linha por obstrução em VS2 (R-05)"
assert resultados_bt[4][1] == SUCCESS and resultados_bt[4][2] == "TRIP_SIS_PARADA" and resultados_bt[4][3] == 0.0, "Tick 5 deve preemptar todas as falhas e executar Trip SIS (R-04)"
assert resultados_bt[4][4] == 2, "Trip SIS no 1º ramo deve avaliar apenas 2 nós (curto-circuito determinístico)"
assert resultados_bt[5][1] == FAILURE and resultados_bt[5][2] == "IDLE_ESPERA", "Tick 6 sem habilitação deve retornar FAILURE seguro"

print("🎯 TODAS AS ASSERÇÕES DA AULA 25 VALIDADAS COM SUCESSO!")

🎯 TODAS AS ASSERÇÕES DA AULA 25 VALIDADAS COM SUCESSO!
